# Cloud013 preflight only

This scaffold supports the design in `design.md`. It does not train, install packages, download weights, attach cloud storage, upload files or change accounts. Run it locally with Python or later in a free Kaggle notebook; free Colab is the fallback. No specific GPU or quota is assumed.

The data gate remains closed: the audited 122 records contain only two distinct verified issues and three rendered patches. This is not a training-ready repair corpus. All code cells use the Python standard library. The final cell refuses training even if its opt-in flag is changed.


In [ ]:
import datetime
import json
import os
import platform
import shutil
import subprocess
from pathlib import Path

# Provider/quota fields require a current manual check in the provider UI.
PROVIDER = "local"  # Later use "kaggle" or "colab" only on that provider.
FREE_TIER_CONFIRMED = False
QUOTA_CHECKED_AT_UTC = None
FREE_GPU_MINUTES_REMAINING = None
OUTPUT_DIRECTORY = Path.cwd()

if PROVIDER not in {"local", "kaggle", "colab"}:
    raise ValueError("Unknown provider")

resource_receipt = {
    "schema_version": "cloud013-preflight-v1",
    "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "provider": PROVIDER,
    "python": platform.python_version(),
    "system": platform.system(),
    "machine": platform.machine(),
    "cpu_count": os.cpu_count(),
    "disk_free_bytes": shutil.disk_usage(OUTPUT_DIRECTORY).free,
    "system_ram_total_bytes": None,
    "system_ram_available_bytes": None,
    "gpu_probe": "unavailable",
    "gpus": [],
    "free_tier_confirmed": FREE_TIER_CONFIRMED,
    "quota_checked_at_utc": QUOTA_CHECKED_AT_UTC,
    "free_gpu_minutes_remaining": FREE_GPU_MINUTES_REMAINING,
    "training_eligible": False,
}

# Linux reports actual currently available memory. Unknown stays unknown.
if Path("/proc/meminfo").is_file():
    memory = {}
    for line in Path("/proc/meminfo").read_text().splitlines():
        name, value = line.split(":", 1)
        memory[name] = int(value.strip().split()[0]) * 1024
    resource_receipt["system_ram_total_bytes"] = memory.get("MemTotal")
    resource_receipt["system_ram_available_bytes"] = memory.get("MemAvailable")
elif hasattr(os, "sysconf"):
    try:
        resource_receipt["system_ram_total_bytes"] = (
            os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
        )
    except (ValueError, OSError):
        pass

# nvidia-smi is a bounded read-only probe, not a CUDA workload.
if shutil.which("nvidia-smi"):
    try:
        probe = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=name,driver_version,memory.total,memory.free",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10, check=False,
        )
        if probe.returncode == 0:
            for line in probe.stdout.splitlines():
                name, driver, total, free = [part.strip() for part in line.rsplit(",", 3)]
                resource_receipt["gpus"].append({
                    "name": name, "driver": driver,
                    "memory_total_mib": int(total), "memory_free_mib": int(free),
                })
            resource_receipt["gpu_probe"] = "reported"
        else:
            resource_receipt["gpu_probe"] = "failed"
    except (OSError, subprocess.TimeoutExpired, ValueError):
        resource_receipt["gpu_probe"] = "failed"

print(json.dumps(resource_receipt, indent=2, allow_nan=False))


## Verify the current audit, without exporting examples

Optionally set `AUDIT_PATH` to a deliberately supplied copy of the public-history audit. No directory is searched. The check verifies the known receipt hash and key counts; it does not qualify data, establish rights or validate a training bundle. A changed audit requires a new review. The current readiness decision cannot be overridden by editing a flag.


In [ ]:
import hashlib

AUDIT_PATH = None  # Explicit Path to a reviewed copy, or leave unset.
EXPECTED_AUDIT_SHA256 = "f06767204bfe38d20b2d410751cc0b8a0748b4e54c706cc9f3494d6af6be77e3"
expected_counts = {
    "public_records": 122,
    "unique_record_tasks": 31,
    "verified_trajectories": 9,
    "verified_unique_tasks": 2,
    "verified_unique_rendered_patches": 3,
}

data_receipt = {
    "schema_version": "cloud013-audit-gate-v1",
    "audit_sha256": EXPECTED_AUDIT_SHA256,
    "audit_checked": False,
    "expected_counts": expected_counts,
    "training_eligible": False,
    "reason": "Current audit is not training-ready; data qualification is required.",
}
if AUDIT_PATH is not None:
    audit_path = Path(AUDIT_PATH)
    if audit_path.is_symlink() or not audit_path.is_file():
        raise ValueError("Audit must be an explicit regular file, not a symlink")
    if audit_path.stat().st_size > 1024 * 1024:
        raise ValueError("Audit exceeds the 1 MiB preflight limit")
    audit_bytes = audit_path.read_bytes()
    if hashlib.sha256(audit_bytes).hexdigest() != EXPECTED_AUDIT_SHA256:
        raise ValueError("Audit changed; a new data and exposure review is required")
    audited = json.loads(audit_bytes)
    if any(audited["audit"].get(key) != value for key, value in expected_counts.items()):
        raise ValueError("Audit counts do not match the reviewed receipt")
    if audited["audit"].get("training_data_ready") is not False:
        raise ValueError("Audit readiness changed; a new protocol review is required")
    if len(audited["exposures"]) != 58 or len(audited["exposed_issue_digests"]) != 45:
        raise ValueError("Exposure inventory changed; a new review is required")
    data_receipt["audit_checked"] = True

print(json.dumps(data_receipt, indent=2, allow_nan=False))


## Review-only training boundary

A later separately reviewed runner must implement full bundle validation, rights/exposure gates, pinned identities, measured fit, complete save/reload and frozen evaluation before training can run. This scaffold has no trainer or model loader. GPU availability and passing the audit-integrity check cannot open that gate. Notebook outputs can be shared with the notebook; review and clear any sensitive output before publication.


In [ ]:
ENABLE_TRAINING = False
if ENABLE_TRAINING:
    raise RuntimeError(
        "Training is unavailable in this scaffold. A separately reviewed runner, "
        "qualified data, exact model/dependency pins, measured fit/resume receipts "
        "and a frozen evaluation protocol are required."
    )
print("Training disabled. No model loaded and no training started.")


## Official references

- [Kaggle notebooks](https://www.kaggle.com/docs/notebooks)
- [Colab FAQ and resource limits](https://research.google.com/colaboratory/faq.html)
- [Hugging Face PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization)
- [Hugging Face Trainer and checkpoint resume](https://huggingface.co/docs/transformers/main_classes/trainer)

Consulted 15 October 2026; recheck current service rules before execution. No fixed free GPU quota, model compatibility or score improvement is claimed.
